In [ ]:

# ============================================================================
# [部分 1] 环境初始化与依赖导入 / Part 1: Environment Setup and Imports
# ============================================================================
import os
from dotenv import load_dotenv
# 引入基础文本模型（这里使用 OpenAI 文本模型，而不是 ChatOpenAI 聊天模型）
# Import base text model (using OpenAI text model, not ChatOpenAI chat model)
from langchain_openai import OpenAI
# PromptTemplate：用于拼装字符串模板
# PromptTemplate: used to assemble string-based prompts
from langchain_core.prompts import PromptTemplate
# CommaSeparatedListOutputParser：将“逗号分隔文本”解析为 Python 列表
# CommaSeparatedListOutputParser: parse comma-separated text into Python list
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 加载本地 .env 中的 API 密钥
# Load API key from local .env file
load_dotenv(override=True)


# ============================================================================
# [部分 2] 初始化模型与输出解析器 / Part 2: Initialize LLM and Output Parser
# ============================================================================

# temperature=0：确定性输出，减少随机发挥，便于稳定解析
# temperature=0: deterministic output, less randomness, easier parsing
llm = OpenAI(temperature=0)

# 创建逗号分隔解析器实例
# Create comma-separated list output parser instance
output_parser = CommaSeparatedListOutputParser()


# ============================================================================
# [部分 3] 自动生成格式约束并展示 / Part 3: Generate and Display Format Instructions
# ============================================================================

# get_format_instructions() 会返回解析器建议的“输出格式规则”
# get_format_instructions() returns parser-recommended output format rules
# 这些规则会嵌入 Prompt 中，约束 LLM 输出形态
# These instructions will be embedded in Prompt to constrain LLM output format
format_instructions = output_parser.get_format_instructions()

print("🤖 The underlying instructions automatically generated by the parser look like this:\n", format_instructions, "\n")
print("-" * 40)


# ============================================================================
# [部分 4] 构建 Prompt 模板（含部分变量）/ Part 4: Build Prompt Template with Partial Variables
# ============================================================================

# partial_variables：把 format_instructions 固定注入模板
# partial_variables: inject fixed format_instructions into the template
# input_variables：运行时只需再提供 things 即可
# input_variables: at runtime, only provide things
prompt = PromptTemplate(
    template="List 3 {things}.\n{format_instructions}",
    input_variables=["things"],
    partial_variables={"format_instructions": format_instructions}
)


# ============================================================================
# [部分 5] 填充动态变量并预览最终 Prompt / Part 5: Fill Runtime Variables and Preview Prompt
# ============================================================================

# 将 things="fruits" 填入模板，得到最终发送给 LLM 的字符串
# Fill things="fruits" into template to get final prompt string sent to LLM
final_prompt = prompt.format(things="fruits")

print("✉️ The complete prompt finally sent to the large model:\n", final_prompt, "\n")
print("-" * 40)


# ============================================================================
# [部分 6] 调用模型获取原始文本输出 / Part 6: Invoke LLM and Get Raw Text Output
# ============================================================================

# OpenAI 文本模型 invoke() 返回的是字符串文本
# OpenAI text model invoke() returns plain string text
output = llm.invoke(final_prompt)

print("📦 Original response from the large model (a string of plain text):")
print(output.strip())
print("Type:", type(output), "\n")
print("-" * 40)


# ============================================================================
# [部分 7] 使用解析器将文本转为结构化数据 / Part 7: Parse Text into Structured Data
# ============================================================================

# parse()：把原始文本按逗号分割并清洗，得到真正的 Python list
# parse(): split and clean raw text by comma to produce a real Python list
things = output_parser.parse(output)

print("✨ Parsed final result (real Python list List):")
print(things)
print("Type:", type(things), "\n")


# ============================================================================
# [部分 8] 按列表方式读取结果 / Part 8: Consume the Parsed List as Native Python Data
# ============================================================================

# 现在 things 已是 Python 列表，可直接按索引访问
# Now things is a Python list; you can directly access by index
print("🍎 First fruit:", things[0])
print("🍌 Second fruit:", things[1])
print("🍊 Third fruit:", things[2])


🤖 解析器自动生成的底层指令长这样：
 Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz` 

----------------------------------------
✉️ 最终发给大模型的完整提示词：
 List 3 fruits.
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz` 

----------------------------------------
📦 大模型的原始回复 (一串纯文本 String):
apple, banana, orange
类型: <class 'str'> 

----------------------------------------
✨ 解析后的最终结果 (真正的 Python 列表 List):
['apple', 'banana', 'orange']
类型: <class 'list'> 

🍎 第一个水果: apple
🍌 第二个水果: banana
🍊 第三个水果: orange
